# Alarm Episode Pipeline: End-to-End Clustering, Filtering & Visualization

Consolidates `create_alarm_clusters_excel.ipynb` + `cluster_visualizations.ipynb` into a single configurable pipeline.

**Pipeline stages:**
1. Load & preprocess events and PV/OP data (with trip filtering)
2. Extract alarm episodes and cluster them → export Excel
3. **Filter stage 1**: Remove alarms < 1 min duration
4. **Filter stage 2**: FI1000 band filtering (with expansion schedule)
5. **Filter stage 3**: Pre-alarm quiet window (adaptive 4h → 3h → 2h → 1h)
6. Funnel summary: alarm counts at each stage
7. Generate per-episode HTML plots + CSV exports

**Configuration**: Edit the config cell below to switch target tags and file paths.


## Section 1: Configuration & Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Edit these to switch target tags and file paths
# ═══════════════════════════════════════════════════════════════════════════════

# ── Target tag and alarm type ──
TARGET_TAG       = '03LIC_1071'   # Tag source name in events data
ALARM_CONDITION  = 'PVLO'         # ConditionName to filter (PVLO, PVHI, etc.)
TARGET_PV_COL    = f'{TARGET_TAG}.PV'

# ── Data file paths ──
EVENTS_FILE           = f'/home/h604827/ControlActions/DATA/trip_filtered_events_dedup.csv'
# EVENTS_FILE           = f'/home/h604827/ControlActions/DATA/combined_events/{TARGET_TAG}_combined_events.parquet'
PV_OP_FILE            = f'/home/h604827/ControlActions/DATA/{TARGET_TAG}_JAN_2026.parquet'
FI1000_PV_DATA_PATH   = f'/home/h604827/ControlActions/DATA/03LIC_1071_JAN_2026_filtered.parquet'
TRIP_FILE             = '/home/h604827/ControlActions/DATA/Final_List_Trip_Duration.csv'
OPERATING_LIMITS_FILE = '/home/h604827/ControlActions/DATA/operating_limits.xlsx'

# ── Trip filtering ──
FILTER_TRIPS = True

# ── Start date cutoff ──
START_DATE = '2022-01-01'

# ── Clustering parameters ──
CLUSTER_GAP_THRESHOLD    = 30   # Minutes gap between alarms to form the same cluster
ACTION_WINDOW_BEFORE     = 240  # Minutes before cluster to capture control actions
ACTION_WINDOW_AFTER      = 60   # Minutes after cluster to capture control actions
MIN_CLUSTER_DURATION_MIN = 2.0  # Minimum cluster total duration (minutes) to keep in Stage 1

# ── FI1000 filtering ──
# Evaluation window matches the plot/action window: [alarm_start - ACTION_WINDOW_BEFORE, alarm_end + ACTION_WINDOW_AFTER]
FI1000_COL             = '02FI_1000.PV'
LOWER_LIMIT_FI1000     = 8.386505779
UPPER_LIMIT_FI1000     = 8.630339925
EXPANSION_SCHEDULE     = [0.0, 0.05, 0.10, 0.15]
FI1000_EXPANSION_USED  = 0.05  # Which expansion fraction to use for stage-2 final filter

# ── Quiet-window look-back for stage-3 ──
QUIET_LOOKBACK_HOURS = [4, 3, 2, 1]   # Try longest first; stop when >= MIN_ALARMS
MIN_ALARMS_THRESHOLD = 30              # Minimum alarms needed to accept a look-back window

# ── Plot / export settings ──
BUFFER_BEFORE_MINUTES = 240   # Minutes before cluster for plot window
BUFFER_AFTER_MINUTES  = 60    # Minutes after cluster for plot window

# ── Filter stages to apply when generating plots ──
# Set to False to skip a stage (its input passes through unchanged to the next stage)
ENABLE_STAGE_1 = False   # Remove clusters < MIN_CLUSTER_DURATION_MIN minutes
ENABLE_STAGE_2 = False   # FI1000 band filtering
ENABLE_STAGE_3 = False   # Pre-alarm quiet window (adaptive look-back)
ENABLE_STAGE_4 = False   # Has OP/SP control actions

# ── Output directory (includes IST run timestamp) ──
_run_ts_ist  = pd.Timestamp.now(tz='Asia/Kolkata').strftime('%d%b%Y_%H%M').upper()
RESULTS_DIR_NAME = f'{TARGET_TAG}_episodes_{_run_ts_ist}'
RESULTS_DIR      = Path(f'/home/h604827/ControlActions/RESULTS/{RESULTS_DIR_NAME}')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_EXCEL = RESULTS_DIR / f'{TARGET_TAG}_{ALARM_CONDITION.lower()}_alarms_clustered_with_control_actions.xlsx'

print(f"Target tag   : {TARGET_TAG} | Condition: {ALARM_CONDITION}")
print(f"Events file  : {EVENTS_FILE}")
print(f"PV/OP file   : {PV_OP_FILE}")
print(f"FI1000 file  : {FI1000_PV_DATA_PATH}")
print(f"Trip filter  : {'ON' if FILTER_TRIPS else 'OFF'}")
print(f"Cluster gap  : {CLUSTER_GAP_THRESHOLD} min | Action window: -{ACTION_WINDOW_BEFORE}/+{ACTION_WINDOW_AFTER} min")
print(f"FI1000 limits: [{LOWER_LIMIT_FI1000}, {UPPER_LIMIT_FI1000}]")
print(f"Output Excel : {OUTPUT_EXCEL}")
print(f"Results dir  : {RESULTS_DIR}")
print()
print(f"Filter stages enabled:")
print(f"  Stage 1 (min duration)   : {'ON' if ENABLE_STAGE_1 else 'OFF (skipped)'}")
print(f"  Stage 2 (FI1000 band)    : {'ON' if ENABLE_STAGE_2 else 'OFF (skipped)'}")
print(f"  Stage 3 (quiet window)   : {'ON' if ENABLE_STAGE_3 else 'OFF (skipped)'}")
print(f"  Stage 4 (OP/SP actions)  : {'ON' if ENABLE_STAGE_4 else 'OFF (skipped)'}")


Target tag   : 03LIC_1071 | Condition: PVLO
Events file  : /home/h604827/ControlActions/DATA/trip_filtered_events_dedup.csv
PV/OP file   : /home/h604827/ControlActions/DATA/03LIC_1071_JAN_2026.parquet
FI1000 file  : /home/h604827/ControlActions/DATA/03LIC_1071_JAN_2026_filtered.parquet
Trip filter  : ON
Cluster gap  : 30 min | Action window: -240/+60 min
FI1000 limits: [8.386505779, 8.630339925]
Output Excel : /home/h604827/ControlActions/RESULTS/03LIC_1071_episodes_01JUN2026_1707/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx
Results dir  : /home/h604827/ControlActions/RESULTS/03LIC_1071_episodes_01JUN2026_1707

Filter stages enabled:
  Stage 1 (min duration)   : OFF (skipped)
  Stage 2 (FI1000 band)    : OFF (skipped)
  Stage 3 (quiet window)   : OFF (skipped)
  Stage 4 (OP/SP actions)  : OFF (skipped)


## Section 2: Load & Preprocess Events Data

In [59]:
def strip_timezone(dt_series):
    """Remove timezone info from a datetime Series if present."""
    if getattr(dt_series.dt, 'tz', None) is not None:
        return dt_series.dt.tz_localize(None)
    return dt_series

# ── Load events ──
if EVENTS_FILE.endswith('.parquet'):
    events_df = pd.read_parquet(EVENTS_FILE)
else:
    events_df = pd.read_csv(EVENTS_FILE, low_memory=False)

events_df['VT_Start'] = pd.to_datetime(events_df['VT_Start'])
events_df['VT_Start'] = strip_timezone(events_df['VT_Start'])
events_df = events_df.sort_values('VT_Start').reset_index(drop=True)

# ── Deduplication ──
dedup_cols = ['VT_Start', 'Source', 'ConditionName', 'Description']
pre_dedup = len(events_df)
events_df = events_df.groupby(dedup_cols, sort=False).first().reset_index()
events_df = events_df.sort_values('VT_Start').reset_index(drop=True)
print(f"Deduplication : {pre_dedup:,} → {len(events_df):,} rows (removed {pre_dedup - len(events_df):,})")

# ── Start-date cutoff ──
events_df = events_df[events_df['VT_Start'] >= pd.to_datetime(START_DATE)].reset_index(drop=True)
print(f"After START_DATE cutoff ({START_DATE}): {len(events_df):,} rows")
print(f"Date range    : {events_df['VT_Start'].min()} → {events_df['VT_Start'].max()}")


Deduplication : 1,364,694 → 1,364,694 rows (removed 0)
After START_DATE cutoff (2022-01-01): 1,245,011 rows
Date range    : 2022-01-01 00:23:50.853000 → 2025-06-27 22:34:17.656000


## Section 3: Load & Preprocess PV/OP Time Series Data

In [60]:
# ── Load PV/OP time series ──
op_pv_data_df = pd.read_parquet(PV_OP_FILE)
if 'TimeStamp' in op_pv_data_df.columns:
    op_pv_data_df['TimeStamp'] = pd.to_datetime(op_pv_data_df['TimeStamp'])
    op_pv_data_df['TimeStamp'] = strip_timezone(op_pv_data_df['TimeStamp'])
    op_pv_data_df.set_index('TimeStamp', inplace=True)
else:
    op_pv_data_df.index = strip_timezone(pd.to_datetime(op_pv_data_df.index))
op_pv_data_df.sort_index(inplace=True)

# ── Identify PV/OP columns ──
tag_cols = [c for c in op_pv_data_df.columns if c.endswith('.PV') or c.endswith('.OP')]
pv_cols  = sorted([c for c in tag_cols if c.endswith('.PV')])
op_cols  = sorted([c for c in tag_cols if c.endswith('.OP')])

print(f"PV/OP data    : {op_pv_data_df.shape[0]:,} rows | {op_pv_data_df.index.min()} → {op_pv_data_df.index.max()}")
print(f"PV columns    : {len(pv_cols)}")
print(f"OP columns    : {len(op_cols)}")

# ── Load operating limits ──
op_limits_raw = pd.read_excel(OPERATING_LIMITS_FILE)
tag_operating_limits = {}
for _, row in op_limits_raw.iterrows():
    tag = row['TagName']
    upper = row['NEW_UPPER_LIMIT']
    lower = row['NEW_LOWER_LIMIT']
    if pd.isna(upper) or (isinstance(upper, str) and 'NOT' in upper.upper()):
        upper = row['OLD_UPPER_LIMIT']
    if pd.isna(lower) or (isinstance(lower, str) and 'NOT' in lower.upper()):
        lower = row['OLD_LOWER_LIMIT']
    try:
        tag_operating_limits[tag] = {'upper': float(upper), 'lower': float(lower)}
    except (ValueError, TypeError):
        pass

TARGET_PV_LOWER = tag_operating_limits.get(TARGET_PV_COL, {}).get('lower', None)
TARGET_PV_UPPER = tag_operating_limits.get(TARGET_PV_COL, {}).get('upper', None)
print(f"Operating limits loaded for {len(tag_operating_limits)} tags")
print(f"{TARGET_PV_COL} limits: lower={TARGET_PV_LOWER}, upper={TARGET_PV_UPPER}")


PV/OP data    : 1,737,586 rows | 2022-01-03 22:45:00 → 2025-06-23 20:44:00
PV columns    : 28
OP columns    : 15
Operating limits loaded for 56 tags
03LIC_1071.PV limits: lower=33.4832240343093, upper=43.8637379407882


## Section 4: Trip Period Filtering

In [61]:
if FILTER_TRIPS:
    trips_df = pd.read_csv(TRIP_FILE)
    trips_df['Stop Date']  = pd.to_datetime(trips_df['Stop Date'])
    trips_df['Start Date'] = pd.to_datetime(trips_df['Start Date'])

    # ── Filter events ──
    pre_ev = len(events_df)
    ev_trip_mask = pd.Series(False, index=events_df.index)
    for _, trip in trips_df.iterrows():
        ev_trip_mask |= (
            (events_df['VT_Start'] >= trip['Stop Date']) &
            (events_df['VT_Start'] <= trip['Start Date'])
        )
    events_df = events_df[~ev_trip_mask].reset_index(drop=True)
    print(f"Trip filter (events)  : {pre_ev:,} → {len(events_df):,} (removed {pre_ev - len(events_df):,})")

    # ── Filter PV/OP data ──
    pre_pv = len(op_pv_data_df)
    pv_trip_mask = pd.Series(False, index=op_pv_data_df.index)
    for _, trip in trips_df.iterrows():
        pv_trip_mask |= (
            (op_pv_data_df.index >= trip['Stop Date']) &
            (op_pv_data_df.index <= trip['Start Date'])
        )
    op_pv_data_df = op_pv_data_df[~pv_trip_mask]
    print(f"Trip filter (PV/OP)   : {pre_pv:,} → {len(op_pv_data_df):,} (removed {pre_pv - len(op_pv_data_df):,})")
    print(f"Trip periods used     : {len(trips_df)}")
else:
    print("Trip filtering is OFF — skipped.")


Trip filter (events)  : 1,245,011 → 1,245,011 (removed 0)
Trip filter (PV/OP)   : 1,737,586 → 1,718,039 (removed 19,547)
Trip periods used     : 106


## Section 5: Extract & Cluster Alarm Episodes

In [62]:
# ── Filter alarm events for target tag ──
pvlo = events_df[
    (events_df['Source'] == TARGET_TAG) &
    (events_df['ConditionName'] == ALARM_CONDITION) &
    (events_df['Category'] == 1)
].copy()

print(f"Target: {TARGET_TAG} | Condition: {ALARM_CONDITION}")
print(f"Total {ALARM_CONDITION} Category=1 events : {len(pvlo)}")
print(f"  Alarm starts (Action NaN/blank)  : {(pvlo['Action'].isna() | (pvlo['Action'] == '')).sum()}")
print(f"  Alarm ends   (Action = OK)       : {(pvlo['Action'] == 'OK').sum()}")
print(f"  Date range   : {pvlo['VT_Start'].min()} → {pvlo['VT_Start'].max()}")

# ── Pair starts with ends to form episodes ──
episodes_list = []
current_start       = None
current_start_value = None

for _, row in pvlo.iterrows():
    is_start = pd.isna(row['Action']) or row['Action'] == ''
    is_end   = row['Action'] == 'OK'

    if is_start and current_start is None:
        current_start       = row['VT_Start']
        current_start_value = row['Value']
    elif is_end and current_start is not None:
        episodes_list.append({
            'alarm_start'  : current_start,
            'alarm_end'    : row['VT_Start'],
            'start_value'  : current_start_value,
            'end_value'    : row['Value'],
        })
        current_start       = None
        current_start_value = None

episodes = pd.DataFrame(episodes_list)
episodes['episode_num']        = range(1, len(episodes) + 1)
episodes['duration_minutes']   = (episodes['alarm_end'] - episodes['alarm_start']).dt.total_seconds() / 60
episodes['gap_to_next_minutes'] = (
    episodes['alarm_start'].shift(-1) - episodes['alarm_end']
).dt.total_seconds() / 60

print(f"\nTotal alarm episodes extracted : {len(episodes)}")
print(f"Duration (min) — mean: {episodes['duration_minutes'].mean():.1f} | "
      f"median: {episodes['duration_minutes'].median():.1f} | "
      f"max: {episodes['duration_minutes'].max():.1f}")


Target: 03LIC_1071 | Condition: PVLO
Total PVLO Category=1 events : 2770
  Alarm starts (Action NaN/blank)  : 1381
  Alarm ends   (Action = OK)       : 1389
  Date range   : 2022-01-05 08:53:41.852900 → 2025-06-22 17:14:56.204800

Total alarm episodes extracted : 1379
Duration (min) — mean: 7.0 | median: 4.6 | max: 495.9


In [63]:
# ── Cluster episodes ──
episodes_all = episodes.copy().reset_index(drop=True)
episodes_all['gap_to_next_minutes'] = (
    episodes_all['alarm_start'].shift(-1) - episodes_all['alarm_end']
).dt.total_seconds() / 60

cluster_id  = 0
cluster_ids = [0]
for g in episodes_all['gap_to_next_minutes'].iloc[:-1]:
    if pd.notna(g) and g <= CLUSTER_GAP_THRESHOLD:
        cluster_ids.append(cluster_id)
    else:
        cluster_id += 1
        cluster_ids.append(cluster_id)

episodes_all['cluster_id'] = cluster_ids

clusters = episodes_all.groupby('cluster_id').agg(
    cluster_start=('alarm_start', 'min'),
    cluster_end  =('alarm_end',   'max'),
    n_alarms     =('episode_num', 'count')
).sort_values('cluster_start')

clusters['total_duration_min']      = (clusters['cluster_end'] - clusters['cluster_start']).dt.total_seconds() / 60
clusters['gap_to_next_cluster_min'] = (
    clusters['cluster_start'].shift(-1) - clusters['cluster_end']
).dt.total_seconds() / 60

def classify_cluster(row):
    if row['n_alarms'] == 1 and row['total_duration_min'] <= 5:
        return 'Isolated brief alarm'
    elif row['n_alarms'] <= 3 and row['total_duration_min'] <= 30:
        return 'Small cluster'
    elif row['total_duration_min'] <= 120:
        return 'Medium situation (<2h)'
    else:
        return 'Extended situation (>2h)'

clusters['cluster_type'] = clusters.apply(classify_cluster, axis=1)

print(f"{len(episodes_all)} raw alarms → {len(clusters)} clusters (gap threshold: {CLUSTER_GAP_THRESHOLD} min)")
print(f"  Single-alarm clusters : {(clusters['n_alarms'] == 1).sum()}")
print(f"  Multi-alarm clusters  : {(clusters['n_alarms'] > 1).sum()}")
print(f"\nCluster type breakdown:")
for ctype in ['Isolated brief alarm', 'Small cluster', 'Medium situation (<2h)', 'Extended situation (>2h)']:
    count = (clusters['cluster_type'] == ctype).sum()
    print(f"  {ctype:30s}: {count}")


1379 raw alarms → 539 clusters (gap threshold: 30 min)
  Single-alarm clusters : 334
  Multi-alarm clusters  : 205

Cluster type breakdown:
  Isolated brief alarm          : 187
  Small cluster                 : 202
  Medium situation (<2h)        : 116
  Extended situation (>2h)      : 34


## Section 6: Export Alarm Clusters Excel

In [64]:
# ── Build output dataframe: one row per alarm with cluster info ──
cluster_id_map = {old: new for new, old in enumerate(sorted(episodes_all['cluster_id'].unique()), 1)}

output = episodes_all[['episode_num', 'alarm_start', 'alarm_end', 'duration_minutes',
                        'gap_to_next_minutes', 'start_value', 'end_value', 'cluster_id']].copy()

output = output.merge(
    clusters[['n_alarms', 'cluster_start', 'cluster_end', 'total_duration_min',
               'gap_to_next_cluster_min', 'cluster_type']],
    left_on='cluster_id', right_index=True
)
output = output.rename(columns={
    'n_alarms'            : 'cluster_total_alarms',
    'cluster_start'       : 'cluster_start_time',
    'cluster_end'         : 'cluster_end_time',
    'total_duration_min'  : 'cluster_total_duration_min',
})
output['cluster_id'] = output['cluster_id'].map(cluster_id_map)
output = output.sort_values('alarm_start').reset_index(drop=True)

# ── Extract control actions per cluster window ──
change_events  = events_df[events_df['ConditionName'] == 'CHANGE'].copy()
cluster_lookup = clusters.copy()
cluster_lookup['cluster_id_new'] = cluster_lookup.index.map(cluster_id_map)
cluster_lookup = cluster_lookup.sort_values('cluster_start').reset_index(drop=True)

WINDOW_BEFORE = pd.Timedelta(minutes=ACTION_WINDOW_BEFORE)
WINDOW_AFTER  = pd.Timedelta(minutes=ACTION_WINDOW_AFTER)

actions_list = []
for _, cl in cluster_lookup.iterrows():
    win_start = cl['cluster_start'] - WINDOW_BEFORE
    win_end   = cl['cluster_end']   + WINDOW_AFTER
    mask      = (change_events['VT_Start'] >= win_start) & (change_events['VT_Start'] <= win_end)
    ca        = change_events[mask].copy()
    ca['cluster_id']    = cl['cluster_id_new']
    ca['cluster_start'] = cl['cluster_start']
    ca['cluster_end']   = cl['cluster_end']
    actions_list.append(ca)

control_actions = pd.concat(actions_list, ignore_index=True)
control_actions = control_actions.sort_values(['cluster_id', 'VT_Start']).reset_index(drop=True)

control_actions['action_timing'] = np.where(
    control_actions['VT_Start'] < control_actions['cluster_start'], 'before',
    np.where(control_actions['VT_Start'] > control_actions['cluster_end'], 'after', 'during')
)
_val  = pd.to_numeric(control_actions['Value'],     errors='coerce')
_prev = pd.to_numeric(control_actions['PrevValue'], errors='coerce')
control_actions['action_direction'] = np.where(
    _val > _prev, 'increase', np.where(_val < _prev, 'decrease', 'no_change')
)

# ── Save Excel ──
actions_save_cols = ['cluster_id', 'cluster_start', 'cluster_end', 'action_timing',
                     'action_direction', 'Source', 'Description', 'VT_Start', 'PrevValue', 'Value']
with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
    output.to_excel(writer, sheet_name='alarm_clusters', index=False)
    control_actions[actions_save_cols].to_excel(writer, sheet_name='control_actions', index=False)

print(f"Saved: {OUTPUT_EXCEL}")
print(f"  alarm_clusters   : {len(output)} rows, {output['cluster_id'].nunique()} clusters")
print(f"  control_actions  : {len(control_actions)} rows, {control_actions['cluster_id'].nunique()} clusters with actions")


Saved: /home/h604827/ControlActions/RESULTS/03LIC_1071_episodes_01JUN2026_1707/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx
  alarm_clusters   : 1379 rows, 539 clusters
  control_actions  : 33721 rows, 499 clusters with actions


## Section 7: Filter Stage 1 — Remove Sub-1-Minute Clusters

> **From this point forward, "alarm" = "cluster".** Each cluster is the unit of interest — individual alarm episodes within a cluster are secondary detail only.

In [65]:
if ENABLE_STAGE_1:
    n_raw_clusters = output['cluster_id'].nunique()
    n_raw_episodes = len(output)

    # Filter at CLUSTER level: keep clusters whose total span (cluster_total_duration_min) >= 1 min.
    # cluster_total_duration_min is the same for all episodes within a cluster, so this is a clean
    # cluster-level predicate. All individual episodes inside a passing cluster are retained.
    filtered_episodes_stage1 = output[output['cluster_total_duration_min'] >= MIN_CLUSTER_DURATION_MIN].copy().reset_index(drop=True)

    n_c1 = filtered_episodes_stage1['cluster_id'].nunique()
    n_s1 = len(filtered_episodes_stage1)

    print("═" * 55)
    print(f"FILTER STAGE 1: Remove clusters < {MIN_CLUSTER_DURATION_MIN} minute(s) duration")
    print("═" * 55)
    print(f"  Raw alarms (clusters)     : {output['cluster_id'].nunique()}")
    print(f"  Removed clusters          : {output['cluster_id'].nunique() - n_c1}")
    print(f"  Remaining alarms          : {n_c1}  (clusters)")
    print(f"  Remaining episodes        : {n_s1}  (individual alarm rows)")
else:
    filtered_episodes_stage1 = output.copy().reset_index(drop=True)
    print("FILTER STAGE 1: SKIPPED — all clusters passed through.")
    print(f"  Clusters : {filtered_episodes_stage1['cluster_id'].nunique()}")
    print(f"  Episodes : {len(filtered_episodes_stage1)}")


FILTER STAGE 1: SKIPPED — all clusters passed through.
  Clusters : 539
  Episodes : 1379


## Section 8: Load FI1000 PV Series & Helper Functions

In [66]:
from IPython.display import display

# ── Load FI1000 PV series ──
fi1000_pv_data = pd.read_parquet(FI1000_PV_DATA_PATH)

if 'TimeStamp' in fi1000_pv_data.columns:
    fi1000_series = fi1000_pv_data[['TimeStamp', FI1000_COL]].copy()
    fi1000_series['TimeStamp'] = pd.to_datetime(fi1000_series['TimeStamp'], errors='coerce')
    fi1000_series = fi1000_series.dropna(subset=['TimeStamp'])
    fi1000_series['TimeStamp'] = strip_timezone(fi1000_series['TimeStamp'])
    fi1000_series = fi1000_series.set_index('TimeStamp')[FI1000_COL]
else:
    fi1000_series = fi1000_pv_data[[FI1000_COL]].copy().reset_index()
    fi1000_series = fi1000_series.rename(columns={fi1000_series.columns[0]: 'TimeStamp'})
    fi1000_series['TimeStamp'] = pd.to_datetime(fi1000_series['TimeStamp'], errors='coerce')
    fi1000_series = fi1000_series.dropna(subset=['TimeStamp'])
    fi1000_series['TimeStamp'] = strip_timezone(fi1000_series['TimeStamp'])
    fi1000_series = fi1000_series.set_index('TimeStamp')[FI1000_COL]

fi1000_series = pd.to_numeric(fi1000_series, errors='coerce').sort_index()
fi1000_series = fi1000_series[~fi1000_series.index.duplicated(keep='last')]

print(f"FI1000 series : {len(fi1000_series):,} rows | {fi1000_series.index.min()} → {fi1000_series.index.max()}")
print(f"FI1000 stats  : min={fi1000_series.min():.4f}, mean={fi1000_series.mean():.4f}, max={fi1000_series.max():.4f}")


# ── Helper functions ──
def get_expanded_fi1000_limits(expansion_fraction):
    """Return (lower, upper) limits expanded symmetrically around the base limits."""
    lower = LOWER_LIMIT_FI1000 * (1 - expansion_fraction)
    upper = UPPER_LIMIT_FI1000 * (1 + expansion_fraction)
    return lower, upper


def evaluate_alarm_window(sample_start, sample_end, lower_limit, upper_limit):
    """
    Check whether the FI1000 signal stays within [lower_limit, upper_limit] for
    every minute in [sample_start, sample_end].

    Returns a pd.Series with keys:
        expected_samples, observed_samples, full_coverage,
        outside_limit_count, min_fi1000, max_fi1000, within_limits
    """
    if pd.isna(sample_start) or pd.isna(sample_end) or sample_end < sample_start:
        return pd.Series({
            'expected_samples'   : 0,
            'observed_samples'   : 0,
            'full_coverage'      : False,
            'outside_limit_count': 0,
            'min_fi1000'         : float('nan'),
            'max_fi1000'         : float('nan'),
            'within_limits'      : False,
        })

    window          = fi1000_series.loc[sample_start:sample_end]
    expected        = int((sample_end - sample_start) / pd.Timedelta(minutes=1)) + 1
    observed        = len(window)
    valid           = window.dropna()
    full_coverage   = (observed == expected) and (len(valid) == expected)

    if valid.empty:
        min_val   = float('nan')
        max_val   = float('nan')
        n_outside = 0
    else:
        min_val   = valid.min()
        max_val   = valid.max()
        n_outside = int((~valid.between(lower_limit, upper_limit, inclusive='both')).sum())

    within = bool(full_coverage and n_outside == 0)

    return pd.Series({
        'expected_samples'   : expected,
        'observed_samples'   : observed,
        'full_coverage'      : full_coverage,
        'outside_limit_count': n_outside,
        'min_fi1000'         : min_val,
        'max_fi1000'         : max_val,
        'within_limits'      : within,
    })

print("Helper functions defined: get_expanded_fi1000_limits(), evaluate_alarm_window()")


FI1000 series : 1,718,039 rows | 2022-01-03 22:45:00 → 2025-06-23 20:44:00
FI1000 stats  : min=0.0000, mean=8.1916, max=10.0000
Helper functions defined: get_expanded_fi1000_limits(), evaluate_alarm_window()


## Section 9: Filter Stage 2 — FI1000 Band Filtering with Expansion Schedule

In [67]:
# Build cluster windows from stage-1 filtered data (one row per cluster)
alarm_windows_s1 = (
    filtered_episodes_stage1[['cluster_id', 'cluster_start_time', 'cluster_end_time']]
    .drop_duplicates(subset=['cluster_id'])
    .dropna(subset=['cluster_start_time', 'cluster_end_time'])
    .rename(columns={'cluster_start_time': 'alarm_start', 'cluster_end_time': 'alarm_end'})
    .copy()
)
alarm_windows_s1 = alarm_windows_s1[alarm_windows_s1['alarm_end'] >= alarm_windows_s1['alarm_start']]
alarm_windows_s1 = alarm_windows_s1.sort_values('alarm_start').reset_index(drop=True)
alarm_windows_s1['alarm_id']         = alarm_windows_s1.index + 1
# Use the same -ACTION_WINDOW_BEFORE / +ACTION_WINDOW_AFTER window as the plot window
alarm_windows_s1['evaluation_start'] = alarm_windows_s1['alarm_start'] - pd.Timedelta(minutes=ACTION_WINDOW_BEFORE)
alarm_windows_s1['evaluation_end']   = alarm_windows_s1['alarm_end']   + pd.Timedelta(minutes=ACTION_WINDOW_AFTER)
alarm_windows_s1['sample_start']     = alarm_windows_s1['evaluation_start'].dt.ceil('min')
alarm_windows_s1['sample_end']       = alarm_windows_s1['evaluation_end'].dt.floor('min')

print(f"Unique alarms (clusters) entering FI1000 filter : {len(alarm_windows_s1)}")
print(f"Evaluation window: [alarm_start - {ACTION_WINDOW_BEFORE} min, alarm_end + {ACTION_WINDOW_AFTER} min]")
print()

summary_rows       = []
fi1000_pass_tables = {}

for expansion_fraction in EXPANSION_SCHEDULE:
    expansion_pct = int(expansion_fraction * 100)
    lower_limit, upper_limit = get_expanded_fi1000_limits(expansion_fraction)

    scenario_eval = alarm_windows_s1.apply(
        lambda row: evaluate_alarm_window(row['sample_start'], row['sample_end'], lower_limit, upper_limit),
        axis=1,
    )
    scenario_results = pd.concat(
        [alarm_windows_s1[['alarm_id', 'cluster_id', 'alarm_start', 'alarm_end',
                             'evaluation_start', 'evaluation_end', 'sample_start', 'sample_end']],
         scenario_eval],
        axis=1,
    )

    passing = scenario_results[scenario_results['within_limits']].copy()
    passing['min_fi1000'] = passing['min_fi1000'].round(6)
    passing['max_fi1000'] = passing['max_fi1000'].round(6)
    fi1000_pass_tables[expansion_pct] = passing.reset_index(drop=True)

    summary_rows.append({
        'scenario'                    : 'base_limits' if expansion_pct == 0 else f'base_plus_{expansion_pct}pct',
        'expansion_pct'               : expansion_pct,
        'lower_limit'                 : round(lower_limit, 6),
        'upper_limit'                 : round(upper_limit, 6),
        'passing_cluster_count'       : len(passing),
        'total_cluster_count'         : len(scenario_results),
        'passing_cluster_pct'         : round(100 * len(passing) / max(len(scenario_results), 1), 2),
        'full_coverage_cluster_count' : int(scenario_results['full_coverage'].sum()),
        'no_full_coverage_count'      : len(scenario_results) - int(scenario_results['full_coverage'].sum()),
    })

fi1000_summary = pd.DataFrame(summary_rows)
print("FI1000 filtering summary (expansion schedule):")
display(fi1000_summary)

# Per-scenario detail tables
for expansion_pct in [int(f * 100) for f in EXPANSION_SCHEDULE]:
    tbl = fi1000_pass_tables[expansion_pct]
    label = 'Base limits' if expansion_pct == 0 else f'Base limits + {expansion_pct}% expansion'
    print(f"\n{label}: {len(tbl)} alarms (clusters) pass FI1000 filter")


Unique alarms (clusters) entering FI1000 filter : 539
Evaluation window: [alarm_start - 240 min, alarm_end + 60 min]

FI1000 filtering summary (expansion schedule):


,scenario,expansion_pct,lower_limit,upper_limit,passing_cluster_count,total_cluster_count,passing_cluster_pct,full_coverage_cluster_count,no_full_coverage_count
0,base_limits,0,8.386506,8.630340,0,539,0.00,482,57
1,base_plus_5pct,5,7.967180,9.061857,172,539,31.91,482,57
2,base_plus_10pct,10,7.547855,9.493374,241,539,44.71,482,57
3,base_plus_15pct,15,7.128530,9.924891,290,539,53.80,482,57



Base limits: 0 alarms (clusters) pass FI1000 filter

Base limits + 5% expansion: 172 alarms (clusters) pass FI1000 filter

Base limits + 10% expansion: 241 alarms (clusters) pass FI1000 filter

Base limits + 15% expansion: 290 alarms (clusters) pass FI1000 filter


In [68]:
if ENABLE_STAGE_2:
    # Select the passing clusters using the configured expansion fraction
    fi1000_expansion_pct = int(FI1000_EXPANSION_USED * 100)
    passing_clusters_s2  = fi1000_pass_tables[fi1000_expansion_pct][['cluster_id', 'alarm_start', 'alarm_end']].copy()

    # Subset stage-1 episodes to only those clusters that pass the FI1000 filter
    filtered_episodes_stage2 = filtered_episodes_stage1[
        filtered_episodes_stage1['cluster_id'].isin(passing_clusters_s2['cluster_id'])
    ].copy().reset_index(drop=True)

    print("═" * 55)
    print(f"FILTER STAGE 2: FI1000 band ({fi1000_expansion_pct}% expansion)")
    print("═" * 55)
    print(f"  Input alarms (clusters)   : {filtered_episodes_stage1['cluster_id'].nunique()}")
    print(f"  Passing alarms (clusters) : {filtered_episodes_stage2['cluster_id'].nunique()}")
    print(f"  Retained episodes         : {len(filtered_episodes_stage2)}")
else:
    fi1000_expansion_pct      = int(FI1000_EXPANSION_USED * 100)
    filtered_episodes_stage2  = filtered_episodes_stage1.copy().reset_index(drop=True)
    print("FILTER STAGE 2: SKIPPED — all clusters passed through.")
    print(f"  Clusters : {filtered_episodes_stage2['cluster_id'].nunique()}")
    print(f"  Episodes : {len(filtered_episodes_stage2)}")


FILTER STAGE 2: SKIPPED — all clusters passed through.
  Clusters : 539
  Episodes : 1379


## Section 10: Filter Stage 3 — Pre-Alarm Quiet Window (Adaptive Look-Back)

In [69]:
if ENABLE_STAGE_3:
    # Unique alarms (clusters) from stage-2, sorted by alarm_start (one row per cluster)
    s2_clusters = (
        filtered_episodes_stage2
        .groupby('cluster_id')
        .agg(alarm_start=('cluster_start_time', 'first'), alarm_end=('cluster_end_time', 'first'))
        .sort_values('alarm_start')
        .reset_index()
    )

    # Check against the stage-2 alarm history: "no other alarm ended within N hours before this alarm's start"
    all_alarm_ends = s2_clusters['alarm_end'].values

    chosen_threshold_hrs = None
    filtered_cluster_ids  = None

    print("═" * 55)
    print("FILTER STAGE 3: Pre-alarm quiet window (adaptive)")
    print("═" * 55)

    for lookback_hrs in QUIET_LOOKBACK_HOURS:
        lookback_td = pd.Timedelta(hours=lookback_hrs)
        quiet_ids   = []

        for _, row in s2_clusters.iterrows():
            window_open  = row['alarm_start'] - lookback_td
            window_close = row['alarm_start']
            # Check if any OTHER alarm (cluster) ended in (window_open, window_close)
            other_ends = s2_clusters[s2_clusters['cluster_id'] != row['cluster_id']]['alarm_end']
            has_prior_alarm = ((other_ends > window_open) & (other_ends < window_close)).any()
            if not has_prior_alarm:
                quiet_ids.append(row['cluster_id'])

        count = len(quiet_ids)
        print(f"  Look-back {lookback_hrs}h : {count} alarms pass (no prior alarm within {lookback_hrs}h before start)")

        if count >= MIN_ALARMS_THRESHOLD:
            chosen_threshold_hrs = lookback_hrs
            filtered_cluster_ids  = quiet_ids
            break

    if filtered_cluster_ids is None:
        # Fallback: use the shortest look-back (1h) even if below threshold
        lookback_hrs = QUIET_LOOKBACK_HOURS[-1]
        lookback_td  = pd.Timedelta(hours=lookback_hrs)
        quiet_ids    = []
        for _, row in s2_clusters.iterrows():
            window_open  = row['alarm_start'] - lookback_td
            window_close = row['alarm_start']
            other_ends   = s2_clusters[s2_clusters['cluster_id'] != row['cluster_id']]['alarm_end']
            if not ((other_ends > window_open) & (other_ends < window_close)).any():
                quiet_ids.append(row['cluster_id'])
        chosen_threshold_hrs = lookback_hrs
        filtered_cluster_ids  = quiet_ids
        print(f"\n  WARNING: All look-backs yielded < {MIN_ALARMS_THRESHOLD} alarms (clusters).")
        print(f"  Using {lookback_hrs}h as fallback with {len(quiet_ids)} alarms.")

    filtered_episodes_stage3 = filtered_episodes_stage2[
        filtered_episodes_stage2['cluster_id'].isin(filtered_cluster_ids)
    ].copy().reset_index(drop=True)

    print(f"\n  Chosen look-back             : {chosen_threshold_hrs} hours")
    print(f"  Alarms after stage 3         : {filtered_episodes_stage3['cluster_id'].nunique()}  (clusters)")
    print(f"  Episodes after stage 3       : {len(filtered_episodes_stage3)}  (individual alarm rows)")
else:
    chosen_threshold_hrs     = None
    filtered_episodes_stage3 = filtered_episodes_stage2.copy().reset_index(drop=True)
    print("FILTER STAGE 3: SKIPPED — all clusters passed through.")
    print(f"  Clusters : {filtered_episodes_stage3['cluster_id'].nunique()}")
    print(f"  Episodes : {len(filtered_episodes_stage3)}")


FILTER STAGE 3: SKIPPED — all clusters passed through.
  Clusters : 539
  Episodes : 1379


## Section 11: Filter Stage 4 — Has OP/SP Control Actions

In [70]:
if ENABLE_STAGE_4:
    # Keep only clusters that have at least one OP or SP action within their window
    # (control_actions was built from the full action window: -ACTION_WINDOW_BEFORE to +ACTION_WINDOW_AFTER)
    clusters_with_op_sp = set(
        control_actions[control_actions['Description'].isin(['OP', 'SP'])]['cluster_id'].unique()
    )

    # Intersect with stage-3 clusters
    s3_cluster_set = set(filtered_episodes_stage3['cluster_id'].unique())
    clusters_with_op_sp_s3 = clusters_with_op_sp & s3_cluster_set

    filtered_episodes_stage4 = filtered_episodes_stage3[
        filtered_episodes_stage3['cluster_id'].isin(clusters_with_op_sp_s3)
    ].copy().reset_index(drop=True)

    print("═" * 55)
    print("FILTER STAGE 4: Has OP/SP control actions")
    print("═" * 55)
    print(f"  Input alarms (clusters)   : {filtered_episodes_stage3['cluster_id'].nunique()}")
    print(f"  Without OP/SP actions     : {filtered_episodes_stage3['cluster_id'].nunique() - len(clusters_with_op_sp_s3)}")
    print(f"  Passing alarms (clusters) : {filtered_episodes_stage4['cluster_id'].nunique()}")
    print(f"  Retained episodes         : {len(filtered_episodes_stage4)}")
else:
    filtered_episodes_stage4 = filtered_episodes_stage3.copy().reset_index(drop=True)
    print("FILTER STAGE 4: SKIPPED — all clusters passed through.")
    print(f"  Clusters : {filtered_episodes_stage4['cluster_id'].nunique()}")
    print(f"  Episodes : {len(filtered_episodes_stage4)}")


FILTER STAGE 4: SKIPPED — all clusters passed through.
  Clusters : 539
  Episodes : 1379


## Section 12: Filtering Funnel Summary

In [71]:
# Before Excel: unit = individual episode
# After  Excel: unit = cluster  (what we call "alarm" from here on)
n_s0 = len(episodes_all);                        n_c0 = episodes_all['cluster_id'].nunique()
n_s1 = len(filtered_episodes_stage1);            n_c1 = filtered_episodes_stage1['cluster_id'].nunique()
n_s2 = len(filtered_episodes_stage2);            n_c2 = filtered_episodes_stage2['cluster_id'].nunique()
n_s3 = len(filtered_episodes_stage3);            n_c3 = filtered_episodes_stage3['cluster_id'].nunique()
n_s4 = len(filtered_episodes_stage4);            n_c4 = filtered_episodes_stage4['cluster_id'].nunique()

def pct(n, total):
    return f"{100 * n / max(total, 1):.1f}%"

def stage_label(name, enabled):
    return name if enabled else f'{name} [SKIPPED]'

s3_label = (f'Stage 3: quiet window ({chosen_threshold_hrs}h)'
            if chosen_threshold_hrs is not None
            else 'Stage 3: quiet window [SKIPPED]')

funnel_rows = [
    ('Raw',                                                                                             n_c0, n_s0, pct(n_c0, n_c0), pct(n_s0, n_s0)),
    (stage_label(f'Stage 1: cluster duration >= {MIN_CLUSTER_DURATION_MIN} min', ENABLE_STAGE_1),      n_c1, n_s1, pct(n_c1, n_c0), pct(n_s1, n_s0)),
    (stage_label(f'Stage 2: FI1000 band ({fi1000_expansion_pct}% exp)',           ENABLE_STAGE_2),      n_c2, n_s2, pct(n_c2, n_c0), pct(n_s2, n_s0)),
    (s3_label,                                                                                          n_c3, n_s3, pct(n_c3, n_c0), pct(n_s3, n_s0)),
    (stage_label('Stage 4: has OP/SP actions',                                    ENABLE_STAGE_4),      n_c4, n_s4, pct(n_c4, n_c0), pct(n_s4, n_s0)),
]
funnel_df = pd.DataFrame(
    funnel_rows,
    columns=['Stage', 'Alarms', 'Episodes', '% of Raw Alarms', '% of Raw Episodes']
)
# "Alarms" = clusters (our working unit); "Episodes" = individual alarm rows (secondary detail)

print("╔══════════════════════════════════════════════════════════════════════════════╗")
print("║                         ALARM FILTERING FUNNEL                             ║")
print("║      Alarms = clusters  |  Episodes = individual alarm rows                ║")
print("╚══════════════════════════════════════════════════════════════════════════════╝")
print()
display(funnel_df)


╔══════════════════════════════════════════════════════════════════════════════╗
║                         ALARM FILTERING FUNNEL                             ║
║      Alarms = clusters  |  Episodes = individual alarm rows                ║
╚══════════════════════════════════════════════════════════════════════════════╝



,Stage,Alarms,Episodes,% of Raw Alarms,% of Raw Episodes
0,Raw,539,1379,100.0%,100.0%
1,Stage 1: cluster duration >= 2.0 min [SKIPPED],539,1379,100.0%,100.0%
2,Stage 2: FI1000 band (5% exp) [SKIPPED],539,1379,100.0%,100.0%
3,Stage 3: quiet window [SKIPPED],539,1379,100.0%,100.0%
4,Stage 4: has OP/SP actions [SKIPPED],539,1379,100.0%,100.0%


## Section 13: Build Tag Color Palette & Ordered Tag List

In [72]:
tag_cols = [c for c in op_pv_data_df.columns if c.endswith('.PV') or c.endswith('.OP')]
pv_cols  = sorted([c for c in tag_cols if c.endswith('.PV')])
op_cols  = sorted([c for c in tag_cols if c.endswith('.OP')])

pv_bases = {c.replace('.PV', ''): c for c in pv_cols}
op_bases = {c.replace('.OP', ''): c for c in op_cols}
all_bases = sorted(set(list(pv_bases.keys()) + list(op_bases.keys())))

# Build ordered tag list: PV before OP per base tag
ordered_tags = []
for base in all_bases:
    if base in pv_bases:
        ordered_tags.append(pv_bases[base])
    if base in op_bases:
        ordered_tags.append(op_bases[base])

# Assign a consistent color per base tag
palette = (
    px.colors.qualitative.Dark24 +
    px.colors.qualitative.Light24 +
    px.colors.qualitative.Alphabet
)
base_colors = {base: palette[i % len(palette)] for i, base in enumerate(all_bases)}

# Auto-detect alarm threshold from events data
TARGET_ALARM_LIMIT = None
if 'AlarmLimit' in events_df.columns:
    alarm_starts = events_df[
        (events_df['Source'] == TARGET_TAG) &
        (events_df['ConditionName'] == ALARM_CONDITION) &
        (events_df['Category'] == 1) &
        (events_df['Action'].isna() | (events_df['Action'] == ''))
    ]
    alarm_limits = alarm_starts['AlarmLimit'].dropna().unique()
    if len(alarm_limits) >= 1:
        TARGET_ALARM_LIMIT = float(alarm_limits[0])
        if len(alarm_limits) > 1:
            print(f"WARNING: Multiple alarm limits found {alarm_limits}, using: {TARGET_ALARM_LIMIT}")
        else:
            print(f"Auto-detected alarm limit: {TARGET_ALARM_LIMIT}")
    else:
        print("Could not auto-detect alarm limit (no matching rows with AlarmLimit)")

target_limits = None
if TARGET_ALARM_LIMIT is not None:
    target_limits = {'lower': TARGET_PV_LOWER, 'upper': TARGET_PV_UPPER, 'alarm': TARGET_ALARM_LIMIT}

print(f"Total tags to plot : {len(ordered_tags)} ({len(pv_cols)} PV + {len(op_cols)} OP)")
print(f"Base tags          : {len(all_bases)}")
print(f"Color palette      : {len(base_colors)} entries")
print(f"Alarm threshold    : {TARGET_ALARM_LIMIT}")


Auto-detected alarm limit: 28.75
Total tags to plot : 43 (28 PV + 15 OP)
Base tags          : 28
Color palette      : 28 entries
Alarm threshold    : 28.75


## Section 14: Define Plot Helper Function

In [73]:
def build_nav_html(all_ids, current_id, id_to_label):
    """Build sticky HTML navigation bar with prev/next buttons and a dropdown."""
    options = []
    for eid in all_ids:
        sel = ' selected' if eid == current_id else ''
        options.append(f'<option value="{eid}"{sel}>{id_to_label[eid]}</option>')

    idx     = all_ids.index(current_id)
    prev_id = all_ids[idx - 1] if idx > 0 else None
    next_id = all_ids[idx + 1] if idx < len(all_ids) - 1 else None

    def btn(label, ep_id, disabled=False):
        if disabled:
            return (f'<button disabled style="padding:6px 14px;border:1px solid #eee;'
                    f'border-radius:4px;background:#f0f0f0;color:#aaa;">{label}</button>')
        return (f'<button onclick="navigateTo({ep_id})" style="padding:6px 14px;cursor:pointer;'
                f'border:1px solid #ccc;border-radius:4px;background:#f8f8f8;">{label}</button>')

    prev_btn = btn(f'◀ Prev ({prev_id})', prev_id) if prev_id else btn('◀ Prev', None, disabled=True)
    next_btn = btn(f'Next ({next_id}) ▶', next_id) if next_id else btn('Next ▶', None, disabled=True)
    options_str = '\n'.join(options)

    return f'''
    <div style="position:sticky;top:0;z-index:9999;background:#fff;padding:10px 15px;
                border-bottom:2px solid #ddd;display:flex;align-items:center;gap:12px;
                font-family:Arial,sans-serif;font-size:14px;">
        <span style="font-weight:bold;color:#333;">Navigate:</span>
        {prev_btn}
        <select id="episode-nav" onchange="navigateTo(this.value)"
                style="padding:6px 10px;border:1px solid #ccc;border-radius:4px;
                       font-size:13px;min-width:350px;">
            {options_str}
        </select>
        {next_btn}
        <span style="color:#888;font-size:12px;margin-left:auto;">
            Episode {idx + 1} of {len(all_ids)}
        </span>
    </div>
    <script>
    function navigateTo(episodeId) {{
        var padded = String(episodeId).padStart(4, '0');
        window.location.href = '../episode_' + padded + '/episode_' + padded + '_plot.html';
    }}
    </script>
    '''


def create_window_plot(window_start, window_end, op_pv_df, ordered_tags, base_colors,
                       title='', actions=None, cluster_regions=None, alarm_regions=None,
                       target_limits=None, operating_limits=None,
                       target_tag=None, target_pv_col=None,
                       fi1000_series=None, fi1000_limits=None):
    """
    Create a normalized PV/OP trend plot with optional control-actions subplot.

    Parameters
    ----------
    window_start, window_end : pd.Timestamp
    op_pv_df                 : DataFrame with TimeStamp index and PV/OP columns
    ordered_tags             : list of column names (PV before OP per base tag)
    base_colors              : dict {base_tag: color_hex}
    title                    : str – plot title
    actions                  : DataFrame with columns [Source, Description, action_direction,
                               VT_Start, PrevValue, Value, action_timing]
    cluster_regions          : list of (cluster_id, start_ts, end_ts) for orange shading
    alarm_regions            : list of (alarm_start, alarm_end, episode_num) for red shading
    target_limits            : dict with keys 'alarm', 'lower', 'upper'
    operating_limits         : dict {tag_name: {'upper': val, 'lower': val}}
    target_tag               : base tag name to highlight (defaults to TARGET_TAG)
    target_pv_col            : full PV column (defaults to TARGET_PV_COL)
    fi1000_series            : pd.Series with DatetimeIndex — FI1000 PV values to overlay
    fi1000_limits            : dict with keys 'lower' and 'upper' for FI1000 band lines
    """
    t_tag  = target_tag    or TARGET_TAG
    t_col  = target_pv_col or TARGET_PV_COL

    mask      = (op_pv_df.index >= window_start) & (op_pv_df.index <= window_end)
    window_df = op_pv_df.loc[mask, [c for c in ordered_tags if c in op_pv_df.columns]].copy()

    if window_df.empty:
        print(f"  No data in window {window_start} → {window_end}")
        return None

    has_actions = actions is not None and len(actions) > 0

    if has_actions:
        n_act_tags   = actions['Source'].nunique()
        act_height   = max(0.15, min(0.35, n_act_tags * 0.04))
        row_heights  = [1 - act_height, act_height]
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                            row_heights=row_heights,
                            subplot_titles=('PV / OP Trends (normalized)', 'Control Actions'))
    else:
        fig = make_subplots(rows=1, cols=1, subplot_titles=('PV / OP Trends (normalized)',))

    x_range = [window_df.index[0], window_df.index[-1]]

    # ── PV/OP traces ──
    for col in ordered_tags:
        if col not in window_df.columns:
            continue
        series = window_df[col]
        if series.isna().all():
            continue

        col_min, col_max = series.min(), series.max()
        normalized = (
            pd.Series(0.5, index=series.index)
            if col_max == col_min
            else (series - col_min) / (col_max - col_min)
        )

        base   = col.rsplit('.', 1)[0]
        suffix = col.rsplit('.', 1)[1]
        color  = base_colors.get(base, '#888888')
        is_op  = suffix == 'OP'
        is_tgt = base == t_tag

        fig.add_trace(go.Scatter(
            x=series.index, y=normalized, mode='lines', name=col,
            legendgroup=base,
            legendgrouptitle_text=base if suffix == 'PV' else None,
            line=dict(color=color, width=2.5 if is_tgt else 1.5,
                      dash='dot' if is_op else 'solid'),
            visible=True if is_tgt else 'legendonly',
            customdata=np.column_stack([series.values]),
            hovertemplate=f'<b>{col}</b><br>Time: %{{x}}<br>Value: %{{customdata[0]:.4f}}<extra></extra>',
        ), row=1, col=1)

        # Per-tag operating limit lines (dashed, same color as PV trace)
        if operating_limits and suffix == 'PV' and col in operating_limits and col_max != col_min:
            lims   = operating_limits[col]
            n_up   = (lims['upper'] - col_min) / (col_max - col_min)
            n_lo   = (lims['lower'] - col_min) / (col_max - col_min)
            for norm_val, hover_label, show_legend in [
                (n_up, f"{col} upper limit ({lims['upper']:.2f})", True),
                (n_lo, f"{col} lower limit ({lims['lower']:.2f})", False),
            ]:
                fig.add_trace(go.Scatter(
                    x=x_range, y=[norm_val, norm_val], mode='lines',
                    name=f"{col} limits ({lims['lower']:.2f}–{lims['upper']:.2f})",
                    legendgroup=base,
                    line=dict(color=color, width=1.2, dash='dash'),
                    visible=True if is_tgt else 'legendonly',
                    showlegend=show_legend,
                    hovertemplate=f'<b>{hover_label}</b><extra></extra>',
                ), row=1, col=1)

    # Alarm threshold horizontal line
    if target_limits and t_col in window_df.columns:
        pv_series = window_df[t_col]
        pv_min, pv_max = pv_series.min(), pv_series.max()
        if pv_max != pv_min:
            alarm_val = target_limits['alarm']
            norm_val  = (alarm_val - pv_min) / (pv_max - pv_min)
            fig.add_trace(go.Scatter(
                x=x_range, y=[norm_val, norm_val], mode='lines',
                name=f'Alarm threshold ({alarm_val})',
                legendgroup=t_tag,
                line=dict(color='red', width=1.5, dash='dash'),
                visible=True,
                hovertemplate=f'<b>Alarm threshold: {alarm_val}</b><extra></extra>',
            ), row=1, col=1)

    # ── FI1000 trace ──
    if fi1000_series is not None:
        fi_window = fi1000_series.loc[window_start:window_end]
        if not fi_window.empty and not fi_window.isna().all():
            fi_min, fi_max = fi_window.min(), fi_window.max()
            if fi_max != fi_min:
                fi_normalized = (fi_window - fi_min) / (fi_max - fi_min)
            else:
                fi_normalized = pd.Series(0.5, index=fi_window.index)

            fi_col_name = fi_window.name if fi_window.name else FI1000_COL
            fig.add_trace(go.Scatter(
                x=fi_window.index, y=fi_normalized, mode='lines',
                name=fi_col_name,
                legendgroup='FI1000',
                legendgrouptitle_text='FI1000',
                line=dict(color='purple', width=2, dash='solid'),
                visible=True,
                customdata=np.column_stack([fi_window.values]),
                hovertemplate=(f'<b>{fi_col_name}</b><br>Time: %{{x}}<br>'
                               f'Value: %{{customdata[0]:.4f}}<extra></extra>'),
            ), row=1, col=1)

            # FI1000 band limit lines
            if fi1000_limits and fi_max != fi_min:
                for limit_val, limit_label, show_leg in [
                    (fi1000_limits['lower'],
                     f"FI1000 lower ({fi1000_limits['lower']:.4f})", True),
                    (fi1000_limits['upper'],
                     f"FI1000 upper ({fi1000_limits['upper']:.4f})", True),
                ]:
                    norm_val = (limit_val - fi_min) / (fi_max - fi_min)
                    fig.add_trace(go.Scatter(
                        x=x_range, y=[norm_val, norm_val], mode='lines',
                        name=limit_label,
                        legendgroup='FI1000',
                        line=dict(color='purple', width=1.2, dash='dash'),
                        visible=True,
                        showlegend=show_leg,
                        hovertemplate=f'<b>{limit_label}</b><extra></extra>',
                    ), row=1, col=1)

    n_rows = 2 if has_actions else 1

    # ── Cluster shading (orange) ──
    if cluster_regions:
        for cid_r, c_start_r, c_end_r in cluster_regions:
            for r in range(1, n_rows + 1):
                fig.add_vrect(x0=c_start_r, x1=c_end_r, fillcolor='rgba(255,165,0,0.10)',
                              line=dict(width=0), layer='below', row=r, col=1)
                fig.add_vline(x=c_start_r, line=dict(color='orange', width=1.5, dash='dash'), row=r, col=1)
                fig.add_vline(x=c_end_r,   line=dict(color='orange', width=1.5, dash='dash'), row=r, col=1)

    # ── Individual alarm shading (red) ──
    if alarm_regions:
        for a_start, a_end, ep_num in alarm_regions:
            for r in range(1, n_rows + 1):
                fig.add_vrect(x0=a_start, x1=a_end, fillcolor='rgba(255,0,0,0.12)',
                              line=dict(color='red', width=0.5, dash='dot'), layer='below', row=r, col=1)

    # ── Control actions subplot ──
    if has_actions:
        sources     = sorted(actions['Source'].unique())
        src_y_map   = {src: i for i, src in enumerate(sources)}

        for _, row in actions.iterrows():
            desc      = str(row['Description'])
            direction = str(row['action_direction'])
            if desc in ('SP', 'OP'):
                color = 'red' if direction == 'decrease' else ('green' if direction == 'increase' else 'grey')
            elif desc == 'MODE':
                color = 'blue'
            else:
                color = 'grey'
            symbol = {'SP': 'diamond', 'OP': 'circle', 'MODE': 'square'}.get(desc, 'x')

            fig.add_trace(go.Scatter(
                x=[row['VT_Start']], y=[src_y_map[row['Source']]], mode='markers',
                marker=dict(color=color, size=10, symbol=symbol, line=dict(width=1, color='darkgrey')),
                showlegend=False,
                hovertemplate=(
                    f'<b>{row["Source"]}</b> ({desc})<br>'
                    f'Time: %{{x}}<br>Direction: {direction}<br>'
                    f'{row["PrevValue"]} → {row["Value"]}<br>'
                    f'Timing: {row["action_timing"]}<extra></extra>'
                ),
            ), row=2, col=1)

        fig.update_yaxes(tickvals=list(src_y_map.values()), ticktext=list(src_y_map.keys()),
                         row=2, col=1, title_text='Tag', gridcolor='rgba(200,200,200,0.3)')

        for label, clr, sym in [
            ('SP Increase', 'green', 'diamond'), ('SP Decrease', 'red', 'diamond'),
            ('OP Increase', 'green', 'circle'),  ('OP Decrease', 'red', 'circle'),
            ('MODE Change', 'blue', 'square'),   ('Other Action', 'grey', 'x'),
        ]:
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode='markers',
                marker=dict(color=clr, size=10, symbol=sym, line=dict(width=1, color='darkgrey')),
                name=label, legendgroup='action_legend', legendgrouptitle_text='Actions',
            ))

    fig.update_layout(
        title=title, height=750 if has_actions else 550, template='plotly_white',
        hovermode='closest',
        legend=dict(groupclick='toggleitem', tracegroupgap=3, font=dict(size=10)),
    )
    fig.update_yaxes(showticklabels=False, title_text='', row=1, col=1)
    return fig

print("create_window_plot() and build_nav_html() defined.")


create_window_plot() and build_nav_html() defined.


## Section 15: Generate Per-Episode HTML Plots & Data Exports

In [74]:
# ── Build per-cluster lookup from stage-4 filtered episodes ──
s3_cluster_ids = sorted(filtered_episodes_stage4['cluster_id'].unique())

# Rebuild cluster_info from the output dataframe (includes individual alarm rows)
s3_cluster_info = {}
for cid in s3_cluster_ids:
    grp = filtered_episodes_stage4[filtered_episodes_stage4['cluster_id'] == cid]
    s3_cluster_info[cid] = {
        'cluster_start' : grp['cluster_start_time'].iloc[0],
        'cluster_end'   : grp['cluster_end_time'].iloc[0],
        'cluster_type'  : grp['cluster_type'].iloc[0],
        'total_alarms'  : grp['cluster_total_alarms'].iloc[0],
        'alarms'        : list(grp[['alarm_start', 'alarm_end', 'episode_num']].itertuples(index=False, name=None)),
    }

# Load control_actions from the saved Excel (ensures column consistency)
actions_from_excel = pd.read_excel(OUTPUT_EXCEL, sheet_name='control_actions')
actions_from_excel['VT_Start']     = pd.to_datetime(actions_from_excel['VT_Start'])
actions_from_excel['cluster_start'] = pd.to_datetime(actions_from_excel['cluster_start'])
actions_from_excel['cluster_end']   = pd.to_datetime(actions_from_excel['cluster_end'])
actions_from_excel['VT_Start']      = strip_timezone(actions_from_excel['VT_Start'])
actions_from_excel['cluster_start'] = strip_timezone(actions_from_excel['cluster_start'])
actions_from_excel['cluster_end']   = strip_timezone(actions_from_excel['cluster_end'])

# Keep only actions whose cluster_id is in the stage-4 set
s3_cluster_actions = {
    cid: grp.copy()
    for cid, grp in actions_from_excel.groupby('cluster_id')
    if cid in s3_cluster_ids
}

# Pre-filter raw CHANGE events (for CSV export and plot refresh)
raw_change_events = events_df[events_df['ConditionName'] == 'CHANGE'].copy()

# Build navigation label map
id_to_label = {
    cid: (f"Episode {cid} | {s3_cluster_info[cid]['total_alarms']} alarm(s) | "
          f"{s3_cluster_info[cid]['cluster_start'].strftime('%Y-%m-%d %H:%M')}")
    for cid in s3_cluster_ids
}

# ── Output directory ──
EPISODE_RESULTS_DIR = RESULTS_DIR / 'episode_visualizations'
EPISODE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Generating plots for {len(s3_cluster_ids)} stage-4 filtered episodes...")
print(f"Target tag  : {TARGET_TAG}")
print(f"Buffer      : -{BUFFER_BEFORE_MINUTES} min / +{BUFFER_AFTER_MINUTES} min")
print(f"Output dir  : {EPISODE_RESULTS_DIR}/")


Generating plots for 539 stage-4 filtered episodes...
Target tag  : 03LIC_1071
Buffer      : -240 min / +60 min
Output dir  : /home/h604827/ControlActions/RESULTS/03LIC_1071_episodes_01JUN2026_1707/episode_visualizations/


In [75]:
for i, cid in enumerate(s3_cluster_ids):
    cdata    = s3_cluster_info[cid]
    cactions = s3_cluster_actions.get(cid, None)

    c_start      = cdata['cluster_start']
    c_end        = cdata['cluster_end']
    window_start = c_start - pd.Timedelta(minutes=BUFFER_BEFORE_MINUTES)
    window_end   = c_end   + pd.Timedelta(minutes=BUFFER_AFTER_MINUTES)

    ep_folder = EPISODE_RESULTS_DIR / f'episode_{cid:04d}'
    ep_folder.mkdir(parents=True, exist_ok=True)

    title = (f'{TARGET_TAG} | Episode {cid} | {cdata["total_alarms"]} alarm(s) | '
             f'{cdata["cluster_type"]} | '
             f'{c_start.strftime("%Y-%m-%d %H:%M")} to {c_end.strftime("%Y-%m-%d %H:%M")}')

    # 1. HTML plot
    fig = create_window_plot(
        window_start, window_end, op_pv_data_df, ordered_tags, base_colors,
        title=title,
        actions=cactions,
        cluster_regions=[(cid, c_start, c_end)],
        alarm_regions=cdata['alarms'],
        target_limits=target_limits,
        operating_limits=tag_operating_limits,
        fi1000_series=fi1000_series,
        fi1000_limits={'lower': LOWER_LIMIT_FI1000, 'upper': UPPER_LIMIT_FI1000},
    )

    if fig is not None:
        plot_html = fig.to_html(include_plotlyjs=True, full_html=True)
        nav_html  = build_nav_html(s3_cluster_ids, cid, id_to_label)
        plot_html = plot_html.replace('<body>', f'<body>\n{nav_html}', 1)
        out_path  = ep_folder / f'episode_{cid:04d}_plot.html'
        with open(out_path, 'w') as f:
            f.write(plot_html)

    # 2. Minutewise PV/OP CSV
    pv_mask   = (op_pv_data_df.index >= window_start) & (op_pv_data_df.index <= window_end)
    pv_window = op_pv_data_df.loc[pv_mask].copy()
    if not pv_window.empty:
        pv_window.to_csv(ep_folder / f'episode_{cid:04d}_pv_data.csv')

    # 3. CHANGE events CSV
    ev_mask   = (raw_change_events['VT_Start'] >= window_start) & (raw_change_events['VT_Start'] <= window_end)
    ep_events = raw_change_events.loc[ev_mask].copy()
    if not ep_events.empty:
        ep_events.to_csv(ep_folder / f'episode_{cid:04d}_events.csv', index=False)

    if (i + 1) % 50 == 0 or (i + 1) == len(s3_cluster_ids):
        print(f"  {i + 1}/{len(s3_cluster_ids)} episodes processed")

print(f"\nDone. All episode data saved to {EPISODE_RESULTS_DIR}/")


  50/539 episodes processed
  100/539 episodes processed
  150/539 episodes processed
  200/539 episodes processed
  250/539 episodes processed
  300/539 episodes processed
  350/539 episodes processed
  400/539 episodes processed
  450/539 episodes processed
  500/539 episodes processed
  539/539 episodes processed

Done. All episode data saved to /home/h604827/ControlActions/RESULTS/03LIC_1071_episodes_01JUN2026_1707/episode_visualizations/


In [76]:
print(f"TARGET_TAG : {TARGET_TAG}")
print(f"Stage 4    : {n_c4} alarms (clusters) with OP/SP actions  |  {n_s4} individual episodes")
print()
print(funnel_df.to_string(index=False))

TARGET_TAG : 03LIC_1071
Stage 4    : 539 alarms (clusters) with OP/SP actions  |  1379 individual episodes

                                         Stage  Alarms  Episodes % of Raw Alarms % of Raw Episodes
                                           Raw     539      1379          100.0%            100.0%
Stage 1: cluster duration >= 2.0 min [SKIPPED]     539      1379          100.0%            100.0%
       Stage 2: FI1000 band (5% exp) [SKIPPED]     539      1379          100.0%            100.0%
               Stage 3: quiet window [SKIPPED]     539      1379          100.0%            100.0%
          Stage 4: has OP/SP actions [SKIPPED]     539      1379          100.0%            100.0%
